# Wikipedia Agent demo

A Pydantic AI agent that answers questions using only English Wikipedia. It searches for page titles, reads pages in chunks, and returns a structured `WikiAnswer` citing the pages it read. See `CONTEXT.md` and `docs/adr/0001-wikipedia-agent-design.md`.

Needs `OPENAI_API_KEY` in `.env` (and optionally `WIKI_AGENT_MODEL`).

In [1]:
import os

os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")

from pydantic_ai import RunUsage

from wiki_agent import create_agent, print_messages

agent = create_agent()

## A single question

The output is a `WikiAnswer`: `answer`, `sources` (pages read with `get_page`) and `found`.

In [2]:
result = await agent.run("Where do capybaras live?")
result.output

WikiAnswer(answer='Capybaras live in all countries in South America except Chile. They are semiaquatic herbivores found near bodies of freshwater, including rivers, lakes, and swamps. Their habitats include savannas, wetlands, and gallery forests. They primarily inhabit lowland areas and can be found at elevations up to 1500 meters in some regions.', sources=[Source(title='Capybara', url='https://en.wikipedia.org/wiki/Capybara')], found=True)

## Inspecting tool calls

`print_messages` shows each tool call with its arguments, so you can see the search → read → answer flow.

In [3]:
print_messages(result.all_messages())

USER: Where do capybaras live?
TOOL CALL: search {"query":"capybara habitat distribution"}
TOOL RETURN: search (978 chars)
TOOL CALL: get_page {"title":"Capybara"}
TOOL RETURN: get_page (8073 chars)
TOOL CALL: get_page {"title":"Capybara","offset":8000}
TOOL RETURN: get_page (8078 chars)
TOOL CALL: get_page {"title":"Capybara","offset":16000}
TOOL RETURN: get_page (3364 chars)
TOOL CALL: final_result {"answer":"Capybaras live in all countries in South America except Chile. They are semiaquatic herbivores found near bodies of freshwater, including rivers, lakes, and swamps. Their habitats include savannas, wetlands, and gallery forests. They primarily inhabit lowland areas and can be found at elevations up to 1500 meters in some regions.","sources":[{"title":"Capybara","url":""}],"found":true}
TOOL RETURN: final_result (23 chars)


## A follow-up question

Passing `message_history` lets the agent see the earlier turn, so "they" resolves to capybaras.

In [4]:
followup = await agent.run("And what do they eat?", message_history=result.all_messages())
followup.output

WikiAnswer(answer='Capybaras are herbivores, grazing mainly on grasses and aquatic plants, as well as fruit and tree bark. They are selective feeders, often choosing specific plant species to eat. During the dry season, they may eat a greater variety of plants compared to the wet season, as fewer plants are available. They also practice coprophagy, eating their own feces to extract additional nutrients and help with digestion.', sources=[Source(title='Capybara', url='https://en.wikipedia.org/wiki/Capybara')], found=True)

## Usage

Each run reports its `RunUsage`; add them up for the whole conversation. Every run is capped by the package's default `UsageLimits`.

In [5]:
usage = RunUsage() + result.usage + followup.usage
usage

RunUsage(cost=Decimal('0.00328425'), details={'reasoning_tokens': 0}, input_tokens=20703, output_tokens=298, requests=6, tool_calls=4)